<a href="https://colab.research.google.com/github/NAJIA2026/NAJIA2026.github.io/blob/main/Analyse_des_consomation_d'energie.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EcoWattSpark") \
    .getOrCreate()

df = spark.read.csv("consommation_ecowatt.csv", header=True, inferSchema=True)

df.printSchema()
print(df.count())
df.show(5, truncate=False)



root
 |-- id_mesure: integer (nullable = true)
 |-- date_heure: timestamp (nullable = true)
 |-- ville: string (nullable = true)
 |-- type_batiment: string (nullable = true)
 |-- conso_kwh: double (nullable = true)
 |-- temperature_ext: double (nullable = true)
 |-- cout_kwh: double (nullable = true)
 |-- nb_personnes: integer (nullable = true)

300
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+
|id_mesure|date_heure         |ville     |type_batiment|conso_kwh|temperature_ext|cout_kwh|nb_personnes|
+---------+-------------------+----------+-------------+---------+---------------+--------+------------+
|1        |2025-02-05 20:00:00|Marrakech |Résidentiel  |7973.1   |35.3           |1.37    |115         |
|2        |2025-02-23 22:00:00|Agadir    |École        |2096.1   |19.6           |1.3     |199         |
|3        |2025-02-17 02:00:00|Tanger    |École        |1720.2   |34.9           |1.2     |167         |
|4        |2025-02

In [ ]:
from pyspark.sql.functions import col

nb_avant = df.count()

df_clean = df.na.drop().filter(
    (col("conso_kwh") >= 0) & (col("conso_kwh") <= 20000) &
    (col("temperature_ext") >= -10) & (col("temperature_ext") <= 50)
)

df_clean = df_clean.withColumn("cout_total", col("conso_kwh") * col("cout_kwh")) \
                   .withColumn("conso_par_personne", col("conso_kwh") / col("nb_personnes"))

nb_apres = df_clean.count()
print(nb_avant)
print(nb_apres)

300
300


In [ ]:
from pyspark.sql.functions import sum, avg, desc

df_clean.groupBy("ville") \
    .agg(sum("conso_kwh").alias("conso_totale")) \
    .show(truncate=False)
df_clean.groupBy("type_batiment") \
    .agg(avg("conso_kwh").alias("conso_moyenne")) \
    .show(truncate=False)
df_clean.groupBy("ville") \
    .agg(sum("conso_kwh").alias("conso_totale")) \
    .orderBy(desc("conso_totale")) \
    .show(truncate=False)

+----------+------------------+
|ville     |conso_totale      |
+----------+------------------+
|Casablanca|217569.00000000003|
|Fès       |209143.39999999997|
|Agadir    |257718.90000000005|
|Rabat     |244939.20000000004|
|Tanger    |335346.9          |
|Marrakech |232741.69999999998|
+----------+------------------+

+-------------+-----------------+
|type_batiment|conso_moyenne    |
+-------------+-----------------+
|Bureau       |5156.969841269843|
|École        |5112.007547169812|
|Résidentiel  |5268.075000000002|
|Hôpital      |4711.804918032788|
|Industriel   |4695.876271186441|
+-------------+-----------------+

+----------+------------------+
|ville     |conso_totale      |
+----------+------------------+
|Tanger    |335346.9          |
|Agadir    |257718.90000000005|
|Rabat     |244939.20000000004|
|Marrakech |232741.69999999998|
|Casablanca|217569.00000000003|
|Fès       |209143.39999999997|
+----------+------------------+



In [ ]:
from pyspark.sql.functions import to_date, hour, date_format, month, sum, avg

df_temp = df_clean.withColumn("date", to_date("date_heure")) \
                  .withColumn("heure", hour("date_heure")) \
                  .withColumn("jour", date_format(to_date("date_heure"), 'EEEE')) \
                  .withColumn("mois", month("date_heure"))

df_temp.groupBy("mois") \
    .agg(sum("conso_kwh").alias("conso_totale_mois")) \
    .orderBy("mois") \
    .show(truncate=False)

df_temp.groupBy("jour") \
    .agg(avg("conso_kwh").alias("conso_moyenne_jour")) \
    .show(truncate=False)

df_temp.groupBy("heure") \
    .agg(avg("conso_kwh").alias("conso_moyenne_heure")) \
    .orderBy("heure") \
    .show(truncate=False)

+----+------------------+
|mois|conso_totale_mois |
+----+------------------+
|1   |457750.5          |
|2   |488114.30000000005|
|3   |551594.3          |
+----+------------------+

+---------+------------------+
|jour     |conso_moyenne_jour|
+---------+------------------+
|Wednesday|5227.132692307694 |
|Tuesday  |5118.782926829269 |
|Friday   |4649.504761904762 |
|Thursday |4903.738636363636 |
|Saturday |4463.758064516128 |
|Monday   |5002.612500000001 |
|Sunday   |5476.811764705883 |
+---------+------------------+

+-----+-------------------+
|heure|conso_moyenne_heure|
+-----+-------------------+
|0    |6133.05            |
|1    |3642.4             |
|2    |4624.241666666667  |
|3    |3408.207692307692  |
|4    |4969.84            |
|5    |3913.891666666666  |
|6    |5513.11818181818   |
|7    |6501.415384615385  |
|8    |5972.026666666668  |
|9    |6239.542857142857  |
|10   |5638.747368421052  |
|11   |6063.293749999999  |
|12   |3878.4500000000007 |
|13   |2629.3749999999995 |

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

df_temp.createOrReplaceTempView("conso")

spark.sql("""
    SELECT type_batiment, SUM(conso_kwh) as conso_totale
    FROM conso
    GROUP BY type_batiment
""").show(truncate=False)

spark.sql("""
    SELECT AVG(conso_par_personne) as moyenne_par_personne
    FROM conso
""").show(truncate=False)

spark.sql("""
    WITH RankedBatiments AS (
        SELECT id_mesure, ville, type_batiment, conso_kwh,
               ROW_NUMBER() OVER (PARTITION BY ville ORDER BY conso_kwh DESC) as rang
        FROM conso
    )
    SELECT ville, id_mesure, type_batiment, conso_kwh
    FROM RankedBatiments
    WHERE rang <= 2
""").show(truncate=False)

+-------------+-----------------+
|type_batiment|conso_totale     |
+-------------+-----------------+
|Bureau       |324889.1000000001|
|École        |270936.4         |
|Résidentiel  |337156.8000000001|
|Hôpital      |287420.1000000001|
|Industriel   |277056.7         |
+-------------+-----------------+

+--------------------+
|moyenne_par_personne|
+--------------------+
|79.4067845188413    |
+--------------------+

+----------+---------+-------------+---------+
|ville     |id_mesure|type_batiment|conso_kwh|
+----------+---------+-------------+---------+
|Agadir    |25       |Hôpital      |9404.3   |
|Agadir    |80       |École        |9350.2   |
|Casablanca|143      |Résidentiel  |9466.5   |
|Casablanca|298      |École        |9448.2   |
|Fès       |272      |Résidentiel  |9507.3   |
|Fès       |63       |Hôpital      |9478.1   |
|Marrakech |121      |École        |9969.1   |
|Marrakech |285      |Bureau       |9877.9   |
|Rabat     |84       |Industriel   |9849.9   |
|Rabat     |1

In [ ]:
from pyspark.sql.functions import when, col, sum, avg

df_niveau = df_temp.withColumn(
    "niveau_conso",
    when(col("conso_kwh") < 500, "Faible")
    .when((col("conso_kwh") >= 500) & (col("conso_kwh") < 2000), "Moyenne")
    .otherwise("Élevée")
)

df_niveau.groupBy("niveau_conso", "type_batiment") \
    .agg(
        sum("cout_total").alias("cout_total"),
        avg("conso_kwh").alias("conso_moyenne")
    ) \
    .orderBy("niveau_conso", "type_batiment") \
    .show(truncate=False)

+------------+-------------+------------------+------------------+
|niveau_conso|type_batiment|cout_total        |conso_moyenne     |
+------------+-------------+------------------+------------------+
|Faible      |Bureau       |258.431           |228.7             |
|Faible      |Hôpital      |2033.6349999999998|307.7             |
|Faible      |Industriel   |1333.1490000000001|383.26666666666665|
|Faible      |Résidentiel  |1255.7250000000001|356.5             |
|Faible      |École        |1484.511          |329.175           |
|Moyenne     |Bureau       |24419.622         |1448.7692307692307|
|Moyenne     |Hôpital      |18554.281         |1470.39           |
|Moyenne     |Industriel   |6773.94           |1061.5            |
|Moyenne     |Résidentiel  |17867.157         |1373.427272727273 |
|Moyenne     |École        |13567.9           |1581.7571428571428|
|Élevée      |Bureau       |362104.752        |6241.355102040817 |
|Élevée      |Hôpital      |325671.29399999994|5895.1673913043